# 06 Per-Cluster: 逐簇深度剖析

面向 **非计算机专业 PI/学生**（ADR-0009）。消费 `06_annotated.ipynb` 的输出，
对 `cell_type_final_v1` 的每个簇单独产出以下内容：

- **UMAP 高亮**（该簇在全局 UMAP 上的位置）
- **Top 标记基因 dotplot**（该簇 vs 其余簇的差异基因）
- **基因集评分小提琴图**（该簇细胞在各细胞类型评分上的分布）
- **跨疾病组丰度柱状图**（该簇在不同 disease/disease_group 中的比例）
- **LLM 叙述段落**（由 LLM 综合以上证据写一段该簇的生物学描述，key 守卫——无 key 跳过）

产物：`results/figures/06b_per_cluster/cluster_{label}.md` + `index.md`

**为什么逐簇做而不是一张总图？** 每个簇的生物学意义需要用文字讲清楚——
这张图解释它在组织中的位置，这段文字解释它的功能、marker 证据、
跨疾病的丰度差异。逐簇 markdown 可以直接作为论文 Results section
的初稿素材，而不是一张看不懂的热图。

**实现纪律（ADR-0003/0009）**：纯 for 循环，无 plugin/registry/class。
判据是非 CS 学生打开 notebook 能否逐行看懂。

### 运行模式

06b 支持两种模式，由 `MODE` 和 `UPSTREAM_PATH` 控制：

**模式 A（全局剖析）**：06 注释后对所有簇做概览式报告
- `UPSTREAM_PATH = "results/06_annotated_v1.h5ad"`
- `LABEL_COL = "cell_type_final_v1"`
- DEG 是簇 vs 全体其他簇

**模式 B（Subset 深度剖析）**：06c subset 重聚类后对精细亚群做深度解读
- `UPSTREAM_PATH = "results/06c_epithelial_subset_v1.h5ad"`（或其他 subset 产物）
- `LABEL_COL = "cell_type_final_subset_v1"`
- DEG 是亚型 vs 同 compartment 其他亚型（更有生物学意义）

典型工作流：先跑模式 A 了解全局 → 决定哪个 compartment 需要细分 → 跑 06c → 回到 06b 用模式 B 深入剖析。


## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：06（注释），读 `06_annotated_v*.h5ad`
- **本 notebook 角色**：只读逐簇深度报告——消费 `06_annotated` 的输出，
  对 `cell_type_final_v1` 的每个簇产出独立 markdown 报告。
- **下游**：无（本 notebook 不产出 h5ad，仅产出 markdown 报告供 PI 审阅）

### 为什么要迭代回跑？
逐簇深度报告是 PI 审阅注释质量的主要入口。当 PI 在上游 06 调整了注释标签
（修改 `marker_assignments` / `pi_decisions`，或换了 `LEIDEN_COL`、`MARKER_CSV`）后，
需要重跑本 notebook 以生成反映最新注释的逐簇报告。

### 如何回跑（两步操作）
1. **改 `UPSTREAM_PATH`**——指向新版 06_annotated 输出
   （例如 `06_annotated_v2.h5ad`）
2. **（可选）改 `OUTPUT_DIR`**——指向新版逐簇报告目录
   （例如 `results/figures/06b_per_cluster_v2/`）避免覆盖旧版报告
   → 重跑本 notebook（Cell → Run All）

### 重要说明
- **本 notebook 是只读分析，不产出 h5ad，不写入 `adata.uns`。**
  因此没有 `stage` / `version` / `upstream` / `status` 追踪字段——这些字段由上游
  `06_annotated` 负责维护。
- 如需追溯"这份逐簇报告是用哪个版本的注释跑的"，检查本 notebook 的
  `UPSTREAM_PATH` 参数，然后查看对应 `.h5ad` 的 `adata.uns["stage"]` 和
  `adata.uns["version"]`。


In [ ]:
# === PARAMS ===
# UPSTREAM_PATH    -- 06_annotated 输出 h5ad（全局模式）或 06c_subset 产物（subset 模式）
# OUTPUT_DIR       -- 逐簇 markdown 输出目录
# LABEL_COL        -- 用哪个 obs 列做逐簇分析
# MODE             -- "global" = 全体分析 | "subset" = compartment 子集分析
# GENESET_CSV      -- 基因集评分用的标记物 CSV（同 06_annotated 的 MARKER_CSV）
# DISEASE_COL      -- 跨疾病丰度分析用哪个 obs 列（如 disease / disease_group）
# N_TOP_GENES      -- 每簇展示 Top 几个标记基因
# VERDICT_MODEL_TIER -- LLM 叙述用的模型档位（从 .env LLM_GROUP 读取，默认 "sonnet"）
# DOMAIN_CONTEXT_PATH -- 领域知识文件路径（可选，LLM prompt 会引用）
# --
# 运行模式说明：
#   全局模式（默认）：UPSTREAM_PATH = "results/06_annotated_v1.h5ad"
#     LABEL_COL = "cell_type_final_v1"
#   Subset 模式：UPSTREAM_PATH = "results/06c_epithelial_subset_v1.h5ad"
#     LABEL_COL = "cell_type_final_subset_v1"
#     MODE = "subset"

UPSTREAM_PATH = "results/06_annotated_v1.h5ad"  # 全局模式：06 产物
# UPSTREAM_PATH = "results/06c_epithelial_subset_v1.h5ad"  # Subset 模式：06c 产物
OUTPUT_DIR    = "results/figures/06b_per_cluster"
LABEL_COL     = "cell_type_final_v1"  # 全局模式用 cell_type_final_v1
# LABEL_COL     = "cell_type_final_subset_v1"  # Subset 模式用 subset 注释列
MODE          = "global"  # "global" = 全体分析 | "subset" = compartment 子集分析
GENESET_CSV   = "references/markers/gastric_epithelial.csv"
DISEASE_COL   = "disease"  # 或 "disease_group"，取决于上游 manifest 的 obs 列
N_TOP_GENES   = 15

# LLM 叙述段落模型档位（从 .env LLM_GROUP 读取对应 tier 的模型名，key 守卫——无 key 跳过）
VERDICT_MODEL_TIER = "sonnet"  # LLM 叙述用的模型档位（sonnet / haiku / opus）
DOMAIN_CONTEXT_PATH = "references/markers/gastric_knowledge_context.md"  # 领域知识文件（可选）


In [ ]:
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
import sys, os, gc
_root = os.getcwd()
if not os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
    _root = os.path.abspath(os.path.join(_root, ".."))
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# 导入（scanpy 原生 + 框架函数）
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime, warnings

from scrna_integration import load_markers

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)

# 加载上游 adata
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

# 列出 obs 中有用的列
_cell_type_cols = [c for c in adata.obs.columns if "cell_type" in c or "leiden" in c]
_score_cols = [c for c in adata.obs.columns if c.startswith("score_")]
print(f"细胞类型相关列: {_cell_type_cols}")
print(f"基因集评分列: {_score_cols}")
# === 模式检测 ===
print(f"\n运行模式: {MODE}")
if MODE == "subset":
    print(f"  子集分析模式——上游为 06c subset 产物")
    print(f"  LABEL_COL = '{LABEL_COL}' (应为 subset 内的精细注释列)")
    print(f"  ⚠️ DEG 对比基准 (reference='rest') 是 subset 内其他簇，不是全体细胞")
else:
    print(f"  全局分析模式——上游为 06 annotated 产物")
    print(f"  LABEL_COL = '{LABEL_COL}'")

# === 领域知识文件加载 ===
_domain_context = ""
if os.path.exists(DOMAIN_CONTEXT_PATH):
    with open(DOMAIN_CONTEXT_PATH, "r") as f:
        _domain_context = f.read()[:3000]  # 截断到 3000 字符避免 prompt 过长
    print(f"\n✓ 领域知识文件加载: {DOMAIN_CONTEXT_PATH} ({len(_domain_context)} chars)")
else:
    print(f"\n△ 领域知识文件不存在: {DOMAIN_CONTEXT_PATH}")
    print(f"  → 可选：创建该文件提供胃粘膜细胞类型定义/最新 marker 共识/转化阶段描述")
    print(f"  → LLM 将仅依赖训练数据知识")
    # 创建模板文件（PI 可按需编辑补充）
    os.makedirs(os.path.dirname(DOMAIN_CONTEXT_PATH), exist_ok=True)
    with open(DOMAIN_CONTEXT_PATH, "w") as f:
        f.write('''# 胃粘膜细胞类型知识库（PI 维护）

## 主要细胞类型及其标志性 marker

### 上皮
- **表面黏液细胞 (Surface mucous)**: MUC5AC, TFF1, GKN1, GKN2
- **颈黏液细胞 (Neck mucous)**: MUC6, TFF2
- **壁细胞 (Parietal)**: ATP4A, ATP4B, GIF
- **主细胞 (Chief)**: PGA3, PGA4, PGA5, LIPF, PGC
- **SPEM (化生前体)**: TFF2, WFDC2, CD44v9, AQP5, CFTR
- **肠化 (Intestinal metaplasia)**: CDX2, MUC2, TFF3, VIL1, OLFM4
- **内分泌细胞**: CHGA, CHGB, SYP

### 转化轴定义
- **Chief → SPEM**: PGA3/PGA4/GIF 下降 + TFF2/WFDC2 上升
- **SPEM → IM**: TFF2/MUC6 下降 + CDX2/MUC2/TFF3 上升
- **Normal → Atrophy**: ATP4A/ATP4B/GKN1 下降 + TFF1/MUC5AC 维持或代偿上升

## 关键参考文献
- Bockerstett 2020: SPEM 定义
- Weis 2022: 胃粘膜单细胞图谱
- Nowicki-Osuch 2023: 胃癌前病变多组学

（PI 按需补充最新发现）
''')
    print(f"  已创建模板: {DOMAIN_CONTEXT_PATH}（PI 可编辑补充）")


## 验证标签列 + 准备输出目录

**为什么先验证？** 如果 LABEL_COL 不存在，直接报错比跑一半才发现更友好。
同时确保标签列是 categorical 类型——scanpy 的 `cat.categories` 依赖这个。

In [ ]:
# 验证标签列存在且非空。
if LABEL_COL not in adata.obs.columns:
    raise KeyError(
        f"LABEL_COL='{LABEL_COL}' 不在 adata.obs.columns 中。"
        f"可用列: {sorted(adata.obs.columns.tolist())}"
    )

# 确保是 categorical（否则 .cat.categories 不可用）
if not hasattr(adata.obs[LABEL_COL], "cat") or not pd.api.types.is_categorical_dtype(adata.obs[LABEL_COL]):
    adata.obs[LABEL_COL] = adata.obs[LABEL_COL].astype("category")

_cluster_ids = sorted(adata.obs[LABEL_COL].cat.categories)
# 排除 NaN 类别（PI 可能部分簇未标注）
_cluster_ids = [c for c in _cluster_ids if pd.notna(c) and str(c) != "nan"]
_cluster_sizes = {c: (adata.obs[LABEL_COL] == c).sum() for c in _cluster_ids}

print(f"标签列: {LABEL_COL}")
print(f"有效簇数: {len(_cluster_ids)}")
for c in _cluster_ids:
    print(f"  {c}: {_cluster_sizes[c]:,} 细胞")

# 加载标记物库（供基因集评分）
_markers = load_markers(GENESET_CSV)
print(f"\n标记物库: {GENESET_CSV} ({len(_markers)} 种细胞类型)")

# 确认上游 embedding 存在（UMAP 绘图需要）
if "X_umap" not in adata.obsm:
    print("\n⚠ 警告: adata.obsm 中无 X_umap，UMAP 高亮图将跳过。"
          "请确保上游 04 已计算 UMAP 坐标。")

# 确认 DISEASE_COL 存在
_disease_available = DISEASE_COL in adata.obs.columns
if _disease_available:
    _diseases = sorted(adata.obs[DISEASE_COL].dropna().unique())
    print(f"\n疾病列 '{DISEASE_COL}' 可用: {len(_diseases)} 种 ({_diseases})")
else:
    print(f"\n⚠ 疾病列 '{DISEASE_COL}' 不存在——跨疾病丰度图将跳过。"
          f"可用 obs 列: {sorted(adata.obs.columns.tolist())}")

## 逐簇循环体

对每个细胞类型标签，执行以下步骤生成一份 markdown：

1. **UMAP 高亮**：`sc.pl.umap` 加 `groups=` 参数，高亮该簇细胞在全局 UMAP 中的位置
2. **Top 标记基因**：`sc.tl.rank_genes_groups` 找该簇 vs 其余簇的差异基因，画 dotplot
3. **基因集评分小提琴**：将该簇细胞的 `score_{celltype}` 列画小提琴图
4. **跨疾病丰度**：将该簇在不同疾病组中的细胞比例画柱状图
5. **LLM 叙述**（可选，key 守卫）：调用 LLM 综合以上证据写一段生物学描述

**为什么在循环内调 `sc.tl.rank_genes_groups`？**
整个 adata 已在上游 06 做了全局 rank_genes_groups。
但这里我们要的是 **该簇 vs 其余所有簇** 的对比（用 `reference="rest"`），
每次子集 mask 不同，所以必须在循环内独立调用。
这是 scanpy 原生操作——没有额外抽象。

In [ ]:
# 逐簇剖析：for 循环遍历每个细胞类型标签。
# 为什么用 for 循环而非 sweep/plugin？见 ADR-0003/0009——
# 非 CS 学生一眼看懂 for 循环，但畏惧回调/注册中心。
import json

# 预处理：如果有 score 列，收集列名
_score_cols_all = [c for c in adata.obs.columns if c.startswith("score_")]
# 确保至少有一个基础 embedding 可用于 UMAP
_has_umap = "X_umap" in adata.obsm

# 如果无 UMAP，尝试在当前 obsm 中找一个 X_ 开头的做 fallback
_umap_key = "X_umap"
if not _has_umap:
    _umap_keys = [k for k in adata.obsm.keys() if k.startswith("X_")]
    if _umap_keys:
        _umap_key = _umap_keys[0]
        _has_umap = True
        print(f"UMAP 不可用，改用 {_umap_key} 坐标画散点图（非标准 UMAP 投影）\n")

# === 计算各簇嵌入 centroid（为邻居对比 DEG 做准备）===
# 为什么在主循环前预先算 centroid？避免循环内重复计算所有簇的 centroid，
# 只需在循环内对当前簇找最近邻居即可。
_pairwise_available = False
_use_rep = None
_centroids = {}
if len(_cluster_ids) > 2:  # 至少 3 个簇才有意义
    for _rep_cand in ["X_pca_harmony", "X_scVI", "X_pca"]:
        if _rep_cand in adata.obsm:
            _use_rep = _rep_cand
            break
    if _use_rep:
        for _c in _cluster_ids:
            _mask_c = adata.obs[LABEL_COL] == _c
            _centroids[_c] = adata.obsm[_use_rep][_mask_c].mean(axis=0)
        _pairwise_available = True
        print(f"\n邻居簇对比 DEG 已启用 (嵌入: {_use_rep}, {len(_centroids)} 个 centroid)")
    else:
        print("\n邻居簇对比 DEG 跳过: 无可用的嵌入坐标 (X_pca_harmony/X_scVI/X_pca)")
else:
    print(f"\n邻居簇对比 DEG 跳过: 簇数不足 ({len(_cluster_ids)} <= 2)")

# 收集每簇的摘要，用于 index.md（改为 dict 供 LLM 步骤按 cid 索引）
_cluster_summaries = {}


for _cid in _cluster_ids:
    _cid_str = str(_cid).replace("/", "_").replace(" ", "_")
    _mask = adata.obs[LABEL_COL] == _cid
    _n = _mask.sum()
    _out_md = os.path.join(OUTPUT_DIR, f"cluster_{_cid_str}.md")
    print(f"\n{'='*60}")
    print(f"处理簇 '{_cid}': {_n:,} 细胞")
    print(f"{'='*60}")

    # ---- 打开该簇的 markdown 输出文件 ----
    _md_lines = []
    _md_lines.append(f"# 簇: {_cid}\n")
    _md_lines.append(f"**标签列**: `{LABEL_COL}`\n")
    _md_lines.append(f"**细胞数**: {_n:,} ({_n/adata.n_obs*100:.1f}%)\n")
    _md_lines.append(f"**上游**: `{UPSTREAM_PATH}`\n")

    # ---- 1. UMAP 高亮 ----
    _md_lines.append(f"\n## 1. UMAP 高亮\n")
    if _has_umap:
        # 创建 highlight 列：该簇为实际标签，其余为 "Other"
        # 为什么用 copy 新列？不改动原始 adata.obs，隔离副作用
        _highlight_col = f"_highlight_{_cid_str}"
        adata.obs[_highlight_col] = "Other"
        adata.obs.loc[_mask, _highlight_col] = str(_cid)
        # 该簇的颜色用红色突出，其余灰色
        _palette = {str(_cid): "#e74c3c", "Other": "#bdc3c7"}

        _fig, _ax = plt.subplots(figsize=(10, 8))
        # 检查 UMAP 坐标是否确实可用——如果 fallback 到非 UMAP 嵌入，
        # sc.pl.umap 硬依赖 X_umap 会崩溃；改用 sc.pl.embedding 传 basis
        if "X_umap" in adata.obsm:
            sc.pl.umap(
                adata, color=_highlight_col, ax=_ax,
                palette=_palette, title=f"UMAP: {_cid} ({_n:,} cells)",
                legend_loc="right margin", show=False,
            )
        else:
            sc.pl.embedding(
                adata, basis=_umap_key, color=_highlight_col, ax=_ax,
                palette=_palette, title=f"{_umap_key}: {_cid} ({_n:,} cells)",
                legend_loc="right margin", show=False,
            )
        _fname = os.path.join(OUTPUT_DIR, f"umap_{_cid_str}.png")
        _fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.close(_fig)
        _md_lines.append(f"![UMAP {_cid}](umap_{_cid_str}.png)\n")
        # 清理临时列
        del adata.obs[_highlight_col]
        print(f"  UMAP 高亮已保存: {_fname}")
    else:
        _md_lines.append("(UMAP 坐标不可用，跳过)\n")
        print("  UMAP 跳过: 无可用的嵌入坐标")

    # ---- 2. Top 标记基因 dotplot ----
    _md_lines.append(f"\n## 2. Top 标记基因\n")
    # 在 adata 上用 mask 做 rank_genes_groups
    # 注意：rank_genes_groups 会写入 adata.uns，每簇覆盖前一次的 key
    sc.tl.rank_genes_groups(
        adata, groupby=LABEL_COL, groups=[_cid],
        reference="rest", method="wilcoxon", n_genes=N_TOP_GENES,
        key_added="rank_genes_per_cluster",
    )
    _df = sc.get.rank_genes_groups_df(adata, group=_cid, key="rank_genes_per_cluster")
    _top_genes = _df["names"].head(N_TOP_GENES).tolist()
    _top_scores = _df["scores"].head(N_TOP_GENES).tolist()
    _top_logfc = _df["logfoldchanges"].head(N_TOP_GENES).tolist()
    _top_pvals = _df["pvals_adj"].head(N_TOP_GENES).tolist()

    _md_lines.append(f"_Top {len(_top_genes)} 差异基因（Wilcoxon, {_cid} vs rest）_\n\n")
    _md_lines.append("| 排名 | 基因 | logFC | -log10(p_adj) |\n")
    _md_lines.append("|------|------|-------|---------------|\n")
    for i, (g, lfc, p) in enumerate(zip(_top_genes, _top_logfc, _top_pvals)):
        _nlogp = -np.log10(max(p, 1e-300))
        _md_lines.append(f"| {i+1} | {g} | {lfc:.2f} | {_nlogp:.1f} |\n")

    # 画 dotplot（该簇 vs 其余，用 top 基因）
    # dotplot 不依赖 UMAP 坐标，只依赖基因表达矩阵和分组
    if _top_genes:
        _fig, _ax = plt.subplots(figsize=(max(6, len(_top_genes)*0.4), 3))
        sc.pl.dotplot(
            adata, var_names=_top_genes, groupby=LABEL_COL,
            standard_scale="var", title=f"Top markers: {_cid}",
            show=False, ax=_ax,
        )
        _fname = os.path.join(OUTPUT_DIR, f"dotplot_{_cid_str}.png")
        _fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.close(_fig)
        _md_lines.append(f"\n![Dotplot {_cid}](dotplot_{_cid_str}.png)\n")
        print(f"  Top 标记基因: {_top_genes}")
    else:
        print(f"  Top 标记基因: (无 DEG 结果)")

# ---- 2b. 邻居簇对比 DEG（精细区分证据）----
    # 为什么加邻居对比？"这个簇跟它最像的簇差在哪"——对连续谱系上的精细区分极其重要
    _pair_genes = []
    _pair_logfc = []
    _nearest = ""
    _pair_flag = False
    if _pairwise_available:
        _current_centroid = _centroids[_cid]
        _distances = {c: np.linalg.norm(_current_centroid - _centroids[c])
                     for c in _cluster_ids if c != _cid}
        _nearest = min(_distances, key=_distances.get)
        # 计算 cluster vs nearest neighbor DEG
        _subset_mask = adata.obs[LABEL_COL].isin([_cid, _nearest])
        if _subset_mask.sum() > 20:
            _adata_pair = adata[_subset_mask].copy()
            sc.tl.rank_genes_groups(
                _adata_pair, groupby=LABEL_COL, groups=[_cid],
                reference=_nearest, method="wilcoxon", n_genes=10,
                key_added="_pairwise_deg",
            )
            _pair_df = sc.get.rank_genes_groups_df(_adata_pair, group=_cid, key="_pairwise_deg")
            _pair_genes = _pair_df["names"].head(10).tolist()
            _pair_logfc = _pair_df["logfoldchanges"].head(10).tolist()
            _pair_flag = True
            del _adata_pair

            _md_lines.append(f"\n## 2b. 与最相似簇的差异（精细区分证据）\n\n")
            _md_lines.append(f"_最相似邻居: {_nearest}（嵌入空间 centroid 距离最近）_\n\n")
            _md_lines.append("| 基因 | logFC (vs neighbour) | 解读方向 |\n")
            _md_lines.append("|------|----------------------|----------|\n")
            for g, lfc in zip(_pair_genes, _pair_logfc):
                _direction = "本簇上调" if lfc > 0 else "本簇下调"
                _md_lines.append(f"| {g} | {lfc:.2f} | {_direction} |\n")
            print(f"  最相似邻居: {_nearest} (距离={_distances[_nearest]:.2f})")
            print(f"  邻居簇对比 DEG: {_pair_genes}")
        else:
            print(f"  邻居簇对比 DEG 跳过: {_cid}+{_nearest} 合计 cell 数不足 ({_subset_mask.sum()} <= 20)")
    else:
        print(f"  邻居簇对比 DEG 跳过: pairwise 不可用")
    # ---- 3. 基因集评分小提琴图 ----
    _md_lines.append(f"\n## 3. 基因集评分\n")
    _score_summary_dict = {}  # 预初始化——避免 NameError（无 score 列时跳过评分后引用）
    if _score_cols_all:
        # 选 Top N 个最相关的评分列（按该簇均值绝对值排序）
        _score_means = {}
        for col in _score_cols_all:
            _score_means[col] = abs(float(adata.obs.loc[_mask, col].mean()))
        _top_scores_sorted = sorted(_score_means, key=_score_means.get, reverse=True)[:10]

        _fig, _axes = plt.subplots(
            max(1, len(_top_scores_sorted) // 5 + 1),
            min(5, len(_top_scores_sorted)),
            figsize=(min(5 * 3, 15), max(3, len(_top_scores_sorted) // 5 * 3)),
        )
        if isinstance(_axes, np.ndarray):
            _axes = _axes.flatten()
        else:
            _axes = [_axes]

        for _ax, _scol in zip(_axes, _top_scores_sorted):
            # 只画该簇细胞的评分分布
            _plot_df = pd.DataFrame({
                "cluster": [str(_cid)] * _n,
                "score": adata.obs.loc[_mask, _scol].values,
            })
            # 小提琴图：该簇细胞在该评分上的分布
            _parts = _ax.violinplot(
                _plot_df["score"].dropna(), positions=[0],
                showmeans=True, showmedians=True,
            )
            _ct_name = _scol.removeprefix("score_")
            _ax.set_title(_ct_name, fontsize=9)
            _ax.set_ylabel("Score")
            _ax.set_xticks([])
            _ax.axhline(y=0, color="gray", linestyle="--", linewidth=0.5)

        # 隐藏多余子图
        for _ax in _axes[len(_top_scores_sorted):]:
            _ax.set_visible(False)

        _fig.suptitle(f"Gene set scores: {_cid} ({_n:,} cells)", fontsize=11)
        _fig.tight_layout()
        _fname = os.path.join(OUTPUT_DIR, f"scores_{_cid_str}.png")
        _fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.close(_fig)

        # 最高分 + 最低分的文字摘要
        _score_summary = []
        for col in _top_scores_sorted[:5]:
            _ct = col.removeprefix("score_")
            _mu = float(adata.obs.loc[_mask, col].mean())
            _pct = float((adata.obs.loc[_mask, col] > 0).mean() * 100)
            _score_summary.append(f"- {_ct}: mean={_mu:.3f}, {_pct:.1f}% cells positive")

        _md_lines.append(f"![Scores {_cid}](scores_{_cid_str}.png)\n\n")
        _md_lines.append("**评分摘要（Top 5）**:\n")
        _md_lines.append("\n".join(_score_summary) + "\n")
        # 构建 score_summary dict（供 LLM 使用——之前只写 markdown，没存结构化数据）
        _score_summary_dict = {}
        for col in _top_scores_sorted[:10]:
            _ct = col.removeprefix("score_")
            _mu = float(adata.obs.loc[_mask, col].mean())
            _pct = float((adata.obs.loc[_mask, col] > 0).mean() * 100)
            _score_summary_dict[_ct] = {"mean": _mu, "pos_pct": _pct}
        print(f"  基因集评分: Top 5 = {_top_scores_sorted[:5]}")
    else:
        _md_lines.append("(无基因集评分列，跳过)\n")
        print("  基因集评分跳过: 无 score_* 列")

    # ---- 4. 跨疾病丰度柱状图 ----
    _md_lines.append(f"\n## 4. 跨疾病丰度\n")
    if _disease_available:
        # 该簇在各疾病组中的细胞比例
        _disease_counts = adata.obs.loc[_mask, DISEASE_COL].value_counts()
        _disease_total = adata.obs[DISEASE_COL].value_counts()
        _disease_pct = (_disease_counts / _disease_total * 100).dropna()

        _fig, _ax = plt.subplots(figsize=(max(6, len(_disease_pct)*0.8), 4))
        _bars = _ax.bar(range(len(_disease_pct)), _disease_pct.values, color="#3498db")
        _ax.set_xticks(range(len(_disease_pct)))
        _ax.set_xticklabels(_disease_pct.index, rotation=45, ha="right", fontsize=9)
        _ax.set_ylabel("% of disease group cells")
        _ax.set_title(f"Abundance of {_cid} across {DISEASE_COL}")
        # 在柱上标注百分比
        for _bar, _val in zip(_bars, _disease_pct.values):
            _ax.text(_bar.get_x() + _bar.get_width()/2, _bar.get_height() + 0.3,
                     f"{_val:.1f}%", ha="center", va="bottom", fontsize=8)
        _fig.tight_layout()
        _fname = os.path.join(OUTPUT_DIR, f"abundance_{_cid_str}.png")
        _fig.savefig(_fname, dpi=150, bbox_inches="tight")
        plt.close(_fig)

        _md_lines.append(f"![Abundance {_cid}](abundance_{_cid_str}.png)\n\n")
        _md_lines.append(f"**{DISEASE_COL} 分布**:\n")
        for _d, _cnt in _disease_counts.items():
            _pct_val = _disease_pct.get(_d, 0)
            _md_lines.append(f"- {_d}: {_cnt} cells ({_pct_val:.1f}% of disease group)\n")
        # 保存该簇的疾病分布 dict（供 LLM 使用）
        _disease_pct_dict = _disease_pct.to_dict() if _disease_available else {}
        print(f"  跨疾病丰度: {dict(_disease_counts)}")
    else:
        _md_lines.append(f"(obs 列 '{DISEASE_COL}' 不存在，跳过)\n")
        # 保存该簇的疾病分布 dict（供 LLM 使用）
        _disease_pct_dict = _disease_pct.to_dict() if _disease_available else {}
        print(f"  跨疾病丰度跳过: DISEASE_COL='{DISEASE_COL}' 不在 obs")

    # ---- 暂时写入 markdown（不含 LLM 段落，LLM 段落单独追加）----
    with open(_out_md, "w") as _f:
        _f.writelines(_md_lines)
    print(f"  基础 markdown 已写入: {_out_md}")

    # 暂存摘要信息和 markdown 路径，供 LLM cell 使用（dict 格式，按 _cid 索引）
    _cluster_summaries[_cid] = {
        "cid": _cid,
        "cid_str": _cid_str,
        "n_cells": _n,
        "top_genes": _top_genes,
        "top_logfc": _top_logfc,
        "out_md": _out_md,
        "disease_abundance": _disease_pct_dict,  # 该簇在各疾病组的百分比
        "score_summary": _score_summary_dict,     # 该簇的基因集评分摘要
        "pair_genes": _pair_genes,                # 邻居簇对比 DEG 基因
        "pair_logfc": _pair_logfc,                # 邻居簇对比 DEG logFC
        "nearest": str(_nearest) if _pair_flag else "",  # 最相似邻居名称
    }

print(f"\n{'='*60}")
print(f"逐簇基础剖析完成。共 {len(_cluster_summaries)} 个簇。")
print(f"{'='*60}")

## LLM 叙述段落（key 守卫——无 key 跳过）

对每个簇，将上述 UMUAP 位置、Top 标记基因、基因集评分、跨疾病丰度综合为
一份 prompt，调用 LLM 写一段 2-3 段的中文生物学叙述。

**为什么用 LLM 写叙述而非手写？** 每簇的证据维度多（marker 列表 +
评分 profile + 疾病分布），人工逐一综合费时且容易遗漏。
LLM 擅长把结构化证据转换为连贯叙述——但它是**初稿生成器**，
PI 仍需审阅修改后再用于论文。

**状态：代码已写，待 PI 在 `.env` 配 key 后人工运行调试。**
无 key 时优雅跳过，notebook 不崩溃。

调用模式沿用 `06_annotated.ipynb` 的 mLLMCelltype 模式：
按模型名前缀路由 provider（openai/anthropic/deepseek/qwen），
从 `.env` 取对应 `{PROVIDER}_API_KEY`。

In [ ]:
# LLM 叙述段落（key 守卫——无 key 跳过）。
# 为什么逐簇单独调用？每个簇的上下文（marker 列表 + 评分 + 疾病分布 + 邻居 DEG）
# 加起来已 ~1000+ tokens，所有簇一起发给 LLM 容易超 context 且输出混杂。
# 逐簇调用虽然 API 调用次数多，但输出结构清晰，每簇叙述独立可审阅。
#
# 调用统一路由：使用 scrna_integration.llm_config（跟 06 一致），
# 从 vault 根 .env 的 LLM_GROUP* schema 读取配置，替代旧的散落 {PROVIDER}_API_KEY 写法。

from scrna_integration.llm_config import load_llm_group_config, get_active_groups

_active_groups = get_active_groups(project_root=_root)
_can_call_llm = len(_active_groups) > 0

if _can_call_llm:
    _cfg = load_llm_group_config(group=None, project_root=_root)  # 默认 group
    _verdict_model = _cfg.get("models", {}).get(VERDICT_MODEL_TIER, "") if _cfg else ""
    _verdict_key = _cfg.get("api_key", "") if _cfg else ""
    _verdict_url = _cfg.get("base_url", "") if _cfg else ""
    _verdict_provider = _cfg.get("provider", "") if _cfg else ""
    print(f"LLM 叙述: provider={_verdict_provider}, model={_verdict_model}")

    for _cid, _summary in _cluster_summaries.items():
        # === 构建完整证据 prompt ===
        _evidence = f"## 簇: {_cid} ({_summary['n_cells']} cells)\n\n"

        # 证据 1: Top DEG
        _evidence += f"### Top 差异表达基因（vs rest）\n{', '.join(_summary['top_genes'])}\n\n"

        # 证据 2: 该簇的疾病分布（修复：之前给的是全局分布）
        _evidence += "### 该簇跨疾病丰度\n"
        for disease, pct in _summary.get("disease_abundance", {}).items():
            _evidence += f"- {disease}: {pct:.1f}%\n"

        # 证据 3: 基因集评分摘要（之前没喂给 LLM）
        if _summary.get("score_summary"):
            _evidence += "\n### 基因集评分（该簇均值 top 5）\n"
            for score_name, val in list(_summary["score_summary"].items())[:5]:
                _evidence += f"- {score_name}: mean={val['mean']:.3f}, 阳性率={val['pos_pct']:.0f}%\n"

        # 证据 4: 转化检测结果（从 06 的 uns 读取，如有）
        _transition = adata.uns.get("transition_detection_v1", {}).get(str(_cid), {})
        if _transition:
            _evidence += f"\n### ⚠️ 转化状态信号\n"
            _evidence += f"- {_transition.get('axis', '')}: {_transition.get('stage', '')} (进度 {_transition.get('progress', '')})\n"

        # 证据 5: 邻居簇对比 DEG（精细区分的核心证据）
        if _summary.get("pair_genes"):
            _evidence += f"\n### 与最相似簇 ({_summary.get('nearest', '')}) 的差异基因\n"
            _up_genes = [g for g, lfc in zip(_summary["pair_genes"], _summary["pair_logfc"]) if lfc > 0]
            _down_genes = [g for g, lfc in zip(_summary["pair_genes"], _summary["pair_logfc"]) if lfc < 0]
            _evidence += f"上调: {', '.join(_up_genes[:5])}\n"
            _evidence += f"下调: {', '.join(_down_genes[:5])}\n"

        # 构建 system + user prompt
        _system = "你是单细胞转录组学专家，专精于人胃粘膜细胞图谱。请结合所有证据分析该细胞簇。"
        if _domain_context:
            _system += f"\n\n参考知识:\n{_domain_context}"

        _user = f'''{_evidence}

请用中文回答以下问题（2-3 段，总计 300-500 字）：
1. 该簇最可能的细胞类型（具体到亚型），为什么这些标记基因支持这个判断
2. 跨疾病丰度差异的生物学意义（如有差异）
3. 该簇是否可能处于某种转化/分化中间态（结合转化检测信号和 marker 特征）
4. 1-2 个值得进一步研究的生物学问题'''

        # === 调用 LLM（统一路由）===
        try:
            if _verdict_provider == "anthropic":
                import requests as _req
                _resp = _req.post(
                    f"{_verdict_url}/v1/messages",
                    headers={"x-api-key": _verdict_key, "anthropic-version": "2023-06-01",
                             "content-type": "application/json"},
                    json={"model": _verdict_model, "max_tokens": 1200,
                          "system": _system,
                          "messages": [{"role": "user", "content": _user}]},
                    timeout=60,
                )
                _data = _resp.json()
                _text_blocks = [b.get("text", "") for b in _data.get("content", [])
                               if b.get("type") == "text"]
                _llm_text = "\n".join(_text_blocks)
            else:
                from openai import OpenAI
                _client = OpenAI(api_key=_verdict_key, base_url=_verdict_url)
                _completion = _client.chat.completions.create(
                    model=_verdict_model, max_tokens=1200, temperature=0.3,
                    messages=[{"role": "system", "content": _system},
                              {"role": "user", "content": _user}]
                )
                _llm_text = _completion.choices[0].message.content

            # 追加到该簇的 markdown
            with open(_summary["out_md"], "a") as f:
                f.write(f"\n\n## 5. LLM 综合叙述\n\n")
                f.write(f"_模型: {_verdict_model} | Provider: {_verdict_provider}_\n\n")
                f.write(_llm_text + "\n")
            print(f"  ✓ Cluster {_cid}: LLM 叙述已追加")

        except Exception as e:
            print(f"  ⚠️ Cluster {_cid}: LLM 调用失败 ({e})")
            with open(_summary["out_md"], "a") as f:
                f.write(f"\n\n## 5. LLM 综合叙述\n\n")
                f.write(f"_(LLM 调用失败: {e}。可用 claude -p 手动分析)_\n")

else:
    print("=" * 60)
    print("LLM 叙述段落已跳过——未检测到 LLM API key。")
    print("请在 vault 根 .env 文件中配置 LLM_GROUP* schema（LLM_DEFAULT_GROUP +")
    print("LLM_GROUP1_PROVIDER / LLM_GROUP1_BASE_URL / LLM_GROUP1_API_KEY /")
    print("LLM_GROUP1_MODEL_SONNET 等），然后重新运行本 cell。")
    print("=" * 60)
    # 为每个 cluster markdown 追加"LLM 未运行"说明
    for _summary in _cluster_summaries.values():
        with open(_summary["out_md"], "a") as f:
            f.write(
                f"\n## 5. LLM 综合叙述\n\n"
                f"_(LLM 未运行——请配置 vault 根 .env 中的 LLM_GROUP* 后重新执行上方 cell)_\n"
            )


## 生成 index.md

将所有逐簇 markdown 串成索引页，方便 PI 快速浏览和跳转。

In [ ]:
# 生成 index.md——将所有逐簇分析页串成一个总表。
_index_path = os.path.join(OUTPUT_DIR, "index.md")
_index_lines = []
_index_lines.append("# Per-Cluster Deep Profile 索引\n\n")
_index_lines.append(f"**标签列**: `{LABEL_COL}`\n")
_index_lines.append(f"**上游**: `{UPSTREAM_PATH}`\n")
_index_lines.append(f"**总细胞数**: {adata.n_obs:,}\n")
_index_lines.append(f"**输出簇数**: {len(_cluster_summaries)}\n\n")
_index_lines.append("| 簇标签 | 细胞数 | 占比 | Top 3 标记基因 | 链接 |\n")
_index_lines.append("|--------|--------|------|----------------|------|\n")

for _cs in _cluster_summaries.values():
    _pct = _cs["n_cells"] / adata.n_obs * 100
    # Top 3 基因（取前三个或更少）
    _top3 = _cs["top_genes"][:3] if len(_cs["top_genes"]) >= 3 else _cs["top_genes"]
    _top3_str = ", ".join(str(g) for g in _top3)
    _link = f"cluster_{_cs['cid_str']}.md"
    _index_lines.append(
        f"| {_cs['cid']} | {_cs['n_cells']:,} | {_pct:.1f}% | {_top3_str} | [{_cs['cid']}]({_link}) |\n"
    )

with open(_index_path, "w") as _f:
    _f.writelines(_index_lines)
print(f"索引页已写出: {_index_path}")
print(f"包含 {len(_cluster_summaries)} 个簇")

## 注：逐簇分析只产生 markdown 产物，不写新 h5ad

`06b_per_cluster` 是只读分析——它消费上游 `06_annotated` 的 h5ad，
产出逐簇 markdown 报告，不修改底层数据。因此不需要 `adata.write_h5ad`。
下面仅做内存释放。

In [ ]:
# 内存纪律自检——只读分析不改变 X 结构，做一个轻量断言即可。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 在只读分析过程中被意外改变: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 保持 sparse CSR float32")

# 释放内存（只读分析不写 h5ad，直接释放即可）
# 为什么不写 h5ad？本 notebook 不产生新数据列——只消费已有 obs/obsm
# 产出是 markdown 文件，已全部写到 OUTPUT_DIR
del adata
del _cluster_summaries
del _markers
gc.collect()
print("内存已释放")